# Leaflet cluster map of talk locations (incremental)

Run this notebook from the `_talks/` directory (where it lives). It:
1. Reads the existing `../talkmap/org-locations.js` as a cache.
2. Scans every `*.md` in this folder for `location:` fields.
3. Geocodes **only** locations that are not already cached.
4. Appends new entries and writes back `../talkmap/org-locations.js`.

Uses Komoot's **Photon** geocoder (free, no API key, lenient rate limits) instead of Nominatim, which has aggressive bans that triggered every time the original notebook re-geocoded all 20+ talks on every run. `map.html` is static Leaflet boilerplate and is not regenerated — only the JS data file changes.


In [1]:
import json, re, time
from pathlib import Path
from geopy.geocoders import Photon

In [2]:
JS_PATH = Path("../talkmap/org-locations.js")
js = JS_PATH.read_text()
m = re.match(r'\s*var\s+addressPoints\s*=\s*(\[.*\])\s*;?\s*$', js, re.DOTALL)
existing = json.loads(m.group(1))
existing_names = set(p[0] for p in existing)
print(f"Cached: {len(existing)} locations")

Cached: 17 locations


In [3]:
talk_locations = []
seen = set()
for md in sorted(Path('.').glob('*.md')):
    text = md.read_text()
    loc_match = re.search(r'location:\s*"([^"]+)"', text)
    if loc_match and loc_match.group(1) not in seen:
        seen.add(loc_match.group(1))
        talk_locations.append(loc_match.group(1))

missing = [loc for loc in talk_locations if loc not in existing_names]
print(f"Talks: {len(talk_locations)} unique locations | Missing: {missing}")

Talks: 17 unique locations | Missing: []


In [4]:
g = Photon(user_agent="papagiannakis.github.io", timeout=15)
new_entries = []
for loc in missing:
    r = g.geocode(loc)
    if r is None:
        print(f"  {loc}: NOT FOUND")
        continue
    new_entries.append([loc, r.latitude, r.longitude])
    print(f"  {loc}: ({r.latitude}, {r.longitude})")
    time.sleep(0.5)

In [5]:
all_points = existing + new_entries
out = "var addressPoints = " + json.dumps(all_points, indent=2) + ";"
JS_PATH.write_text(out)
print(f"Wrote {len(all_points)} points to {JS_PATH}")

Wrote 17 points to ../talkmap/org-locations.js
